# N-gram Tree Speculative Decoding

**Training-free speculative decoding** combining:
- N-gram dictionary lookups for candidate generation
- Tree attention for parallel verification
- Optimized KV cache (no recomputation)

Best for **code generation** where patterns repeat.

In [1]:
!pip install transformers accelerate torch -q

In [2]:
import torch
import torch.nn.functional as F
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

PyTorch: 2.10.0+cu128, CUDA: True


In [2]:
pip show transformers

Name: transformers
Version: 5.0.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer-slim
Required-by: peft, sentence-transformers


## 1. N-gram Dictionary

In [3]:
@dataclass
class Continuation:
    tokens: Tuple[int, ...]
    frequency: int = 1
    last_seen: int = 0

    def score(self, step: int) -> float:
        return self.frequency + 0.1 / (1 + step - self.last_seen)


class NGramDictionary:
    def __init__(self, max_n: int = 6, continuation_length: int = 4):
        self.max_n = max_n
        self.continuation_length = continuation_length
        self._store = defaultdict(list)
        self._step = 0

    def build_from_prompt(self, tokens: List[int]):
        self._extract(tokens)

    def update(self, new_tokens: List[int], context: List[int]):
        self._step += len(new_tokens)
        self._extract(context[-(self.max_n + self.continuation_length):])

    def _extract(self, tokens: List[int]):
        for i in range(len(tokens) - self.continuation_length):
            for slen in range(1, min(self.max_n, i + 2)):
                if i + slen + self.continuation_length > len(tokens):
                    break
                suffix = tuple(tokens[i:i + slen])
                cont = tuple(tokens[i + slen:i + slen + self.continuation_length])
                self._add(suffix, cont)

    def _add(self, suffix, cont):
        for c in self._store[suffix]:
            if c.tokens == cont:
                c.frequency += 1
                c.last_seen = self._step
                return
        if len(self._store[suffix]) < 10:
            self._store[suffix].append(Continuation(cont, 1, self._step))

    def query(self, suffix: List[int], top_k: int = 5) -> List[Tuple[int, ...]]:
        results = []
        seen = set()
        for slen in range(min(len(suffix), self.max_n - 1), 0, -1):
            key = tuple(suffix[-slen:])
            for c in self._store.get(key, []):
                if c.tokens not in seen:
                    seen.add(c.tokens)
                    results.append((c.tokens, c.score(self._step) * (1 + 0.5 * slen)))
        results.sort(key=lambda x: -x[1])
        return [t for t, _ in results[:top_k]]

    def stats(self):
        return {"suffixes": len(self._store), "conts": sum(len(v) for v in self._store.values())}

## 2. Candidate Tree

In [4]:
@dataclass
class TreeNode:
    token_id: int
    node_idx: int
    parent_idx: int
    depth: int
    children: dict = field(default_factory=dict)


class CandidateTree:
    def __init__(self, budget: int = 64):
        self.budget = budget
        self.reset()

    def reset(self):
        self.root = TreeNode(-1, 0, -1, 0)
        self.nodes = [self.root]
        self._counter = 1

    def build(self, continuations: List[Tuple[int, ...]]):
        self.reset()
        for cont in continuations:
            curr = self.root
            for tok in cont:
                if self._counter > self.budget:
                    break
                if tok in curr.children:
                    curr = curr.children[tok]
                else:
                    node = TreeNode(tok, self._counter, curr.node_idx, curr.depth + 1)
                    self._counter += 1
                    curr.children[tok] = node
                    self.nodes.append(node)
                    curr = node
        return self

    def num_nodes(self): return len(self.nodes) - 1
    def get_tokens(self): return [n.token_id for n in self.nodes[1:]]
    def get_depths(self): return [n.depth for n in self.nodes[1:]]

    def get_ancestors(self, idx: int) -> List[int]:
        anc = []
        while idx > 0:
            anc.append(idx)
            idx = self.nodes[idx].parent_idx
        return anc[::-1]

    def get_paths(self) -> List[List[int]]:
        paths = []
        def dfs(node, path):
            if node.token_id >= 0:
                path = path + [node.token_id]
            if not node.children:
                if path: paths.append(path)
            else:
                for c in node.children.values(): dfs(c, path)
        dfs(self.root, [])
        return paths

## 3. Tree Attention + Verification

In [14]:

def generate_tree_buffers(tree: CandidateTree, device="cuda"):
    n = tree.num_nodes()
    if n == 0:
        return {
            "mask": torch.ones(1, 1, 1, 1, device=device),
            "pos": torch.zeros(1, dtype=torch.long, device=device),
            "retrieve": torch.zeros(1, 1, dtype=torch.long, device=device),
            "tokens": torch.tensor([], dtype=torch.long, device=device),
        }

    seq = 1 + n
    mask = torch.eye(seq, dtype=torch.float32)
    mask[:, 0] = 1.0
    for node in tree.nodes[1:]:
        for anc in tree.get_ancestors(node.node_idx):
            mask[node.node_idx, anc] = 1.0

    pos = torch.zeros(seq, dtype=torch.long)
    for i, d in enumerate(tree.get_depths()):
        pos[i + 1] = d

    paths = tree.get_paths()
    max_len = max(len(p) for p in paths) if paths else 0
    retrieve = []
    for path in paths:
        indices = [0]
        curr = tree.root
        for tok in path:
            curr = curr.children[tok]
            indices.append(curr.node_idx)
        indices += [-1] * (max_len + 1 - len(indices))
        retrieve.append(indices)

    return {
        "mask": mask.unsqueeze(0).unsqueeze(0).to(device),
        "pos": pos.to(device),
        "retrieve": torch.tensor(retrieve, dtype=torch.long, device=device),
        "tokens": torch.tensor(tree.get_tokens(), dtype=torch.long, device=device),
    }


def verify_candidates(logits, tokens, retrieve):
    """Find longest accepted path."""
    if tokens.shape[0] == 0:
        return 0, []

    preds = torch.argmax(logits[0], dim=-1)
    best_len, best_tokens = 0, []

    for path in retrieve:
        accepted = []
        for i in range(len(path) - 1):
            node_idx = path[i + 1].item()
            if node_idx < 0:
                break
            pred_pos = path[i].item()
            expected = tokens[node_idx - 1].item()
            if preds[pred_pos].item() == expected:
                accepted.append(expected)
            else:
                break
        if len(accepted) > best_len:
            best_len, best_tokens = len(accepted), accepted

    return best_len, best_tokens


## 4. Optimized KV Cache

In [19]:

class ManagedKVCache:
    """KV cache with crop-based rollback for tree speculation."""

    def __init__(self, model):
        self.model = model
        self.past_kv = None
        self.length = 0

    def reset(self):
        self.past_kv = None
        self.length = 0

    def prefill(self, input_ids):
        self.reset()
        out = self.model(input_ids=input_ids, use_cache=True, return_dict=True)
        self.past_kv = out.past_key_values
        self.length = input_ids.shape[1]
        return out.logits[:, -1, :]

    def forward_single(self, token):
        if token.dim() == 1:
            token = token.unsqueeze(0)
        pos = torch.tensor([[self.length]], device=token.device)
        out = self.model(
            input_ids=token, position_ids=pos,
            past_key_values=self.past_kv, use_cache=True, return_dict=True,
        )
        self.past_kv = out.past_key_values
        self.length += 1
        return out.logits[:, -1, :]

    def forward_tree(self, ids, pos, mask):
        self._pre_tree_length = self.length
        out = self.model(
            input_ids=ids, position_ids=pos, attention_mask=mask,
            past_key_values=self.past_kv, use_cache=True, return_dict=True,
        )
        self.past_kv = out.past_key_values
        return out.logits

    def rollback_and_replay(self, accepted_tokens, device):
        """Crop KV cache back to pre-tree state, replay accepted tokens."""
        self.past_kv.crop(self._pre_tree_length)
        self.length = self._pre_tree_length

        logits = None
        for tok in accepted_tokens:
            logits = self.forward_single(
                torch.tensor([[tok]], device=device)
            )
        return logits


In [25]:
import inspect
print(inspect.signature(CandidateTree.__init__))

(self, budget: int = 64)


## 5. Full Decoder

In [20]:

@dataclass
class DecodingStats:
    total_tokens: int = 0
    total_steps: int = 0
    accepted_counts: dict = field(default_factory=dict)
    fallback_count: int = 0

    def record(self, n, is_fallback=False):
        self.total_tokens += n + 1
        self.total_steps += 1
        self.accepted_counts[n] = self.accepted_counts.get(n, 0) + 1
        if is_fallback:
            self.fallback_count += 1

    @property
    def tokens_per_step(self):
        return self.total_tokens / max(1, self.total_steps)

    @property
    def speculation_rate(self):
        return 1 - (self.fallback_count / max(1, self.total_steps))

    def __repr__(self):
        return (f"Stats(tokens={self.total_tokens}, steps={self.total_steps}, "
                f"tok/step={self.tokens_per_step:.2f}, spec={self.speculation_rate:.0%}, "
                f"accepts={dict(sorted(self.accepted_counts.items()))})")


class NGramSpeculativeDecoder:
    def __init__(self, model, tokenizer, max_n=6, cont_len=4, budget=64, top_k=5):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device
        self.max_n = max_n
        self.cont_len = cont_len
        self.top_k = top_k
        self.ngram = NGramDictionary(max_n, cont_len)
        self.tree = CandidateTree(budget)
        self.kv = ManagedKVCache(model)

    @torch.no_grad()
    def generate(self, input_ids, max_new=100, verbose=False):
        input_ids = input_ids.to(self.device)
        tokens = input_ids[0].tolist()
        self.ngram = NGramDictionary(self.max_n, self.cont_len)
        self.ngram.build_from_prompt(tokens)

        if verbose:
            print(f"Prompt: {len(tokens)} tokens, Dict: {self.ngram.stats()}")

        logits = self.kv.prefill(input_ids)
        stats = DecodingStats()
        gen = 0
        eos = self.tokenizer.eos_token_id

        while gen < max_new:
            curr_id = torch.argmax(logits, dim=-1).item()

            # Query and filter continuations
            suffix = tokens[-(self.max_n - 1):]
            conts = self.ngram.query(suffix, self.top_k)
            filtered = [c[1:] for c in conts if len(c) > 1 and c[0] == curr_id]

            has_tree = filtered and self.tree.build(filtered).num_nodes() > 0

            if not has_tree:
                # === FALLBACK ===
                tokens.append(curr_id)
                gen += 1
                stats.record(0, is_fallback=True)
                if curr_id == eos:
                    break
                logits = self.kv.forward_single(
                    torch.tensor([[curr_id]], device=self.device)
                )
                self.ngram.update([curr_id], tokens[-(self.max_n + self.cont_len):])
                continue

            # === TREE SPECULATION ===
            buf = generate_tree_buffers(self.tree, self.device)
            tree_tok = buf["tokens"]

            # Combined: [curr_token, tree_tokens...]
            combined = torch.cat(
                [torch.tensor([[curr_id]], device=self.device),
                 tree_tok.unsqueeze(0)], dim=1
            )
            tree_len = combined.shape[1]

            # Position IDs
            pos = (buf["pos"] + len(tokens)).unsqueeze(0)

            # Attention mask
            past_len = self.kv.length
            dtype = next(self.model.parameters()).dtype
            attn = torch.zeros(1, 1, tree_len, past_len + tree_len,
                             device=self.device, dtype=dtype)
            tree_attn = torch.where(buf["mask"].bool(), 0.0, float("-inf")).to(dtype)
            attn[:, :, :, past_len:] = tree_attn

            # Tree forward
            out_logits = self.kv.forward_tree(combined, pos, attn)

            # Verify
            accept_n, accepted = verify_candidates(out_logits, tree_tok, buf["retrieve"])

            new_toks = [curr_id] + accepted

            # Rollback and replay — correct KV cache
            logits = self.kv.rollback_and_replay(new_toks, self.device)

            tokens.extend(new_toks)
            gen += len(new_toks)
            stats.record(accept_n)

            if verbose and stats.total_steps % 10 == 0:
                print(f"Step {stats.total_steps}: {stats.tokens_per_step:.2f} tok/step, "
                      f"gen={gen}/{max_new}")

            if eos in new_toks:
                break

            self.ngram.update(new_toks, tokens[-(self.max_n + self.cont_len):])

        if verbose:
            print(f"\nDone: {stats}")

        return torch.tensor([tokens], device=self.device), stats

## Test

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
print("Loaded!")

Loading Qwen/Qwen2.5-Coder-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded!


In [21]:
prompt = '''class User:
      def __init__(self, name, age):
          self.name = name
          self.age = age

  class Admin:
      def __init__(self, name, age):
          self.name = name
          self.age = age

  class Guest:
      def __init__(self, name, age):
          self.name = name
          self.age = age

  class Member:
      def __init__(self, name, age):'''

In [22]:
decoder = NGramSpeculativeDecoder(model, tokenizer)
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

output_ids, stats = decoder.generate(input_ids, max_new=50, verbose=True)
print("\n" + "=" * 50)
print(tokenizer.decode(output_ids[0]))
print("\n" + "=" * 50)
print(f"Stats: {stats}")

Prompt: 113 tokens, Dict: {'suffixes': 270, 'conts': 372}
Step 20: 2.55 tok/step, gen=51/50

Done: Stats(tokens=51, steps=20, tok/step=2.55, spec=60%, accepts={0: 9, 1: 1, 3: 10})

class User:                                                                                                                       
      def __init__(self, name, age):     
          self.name = name                                                                                                                      
          self.age = age                                                                                                                        
                                                                                                                                                
  class Admin:                                              
      def __init__(self, name, age):
          self.name = name                                                                                       

In [23]:
# Compare with baseline
print("Baseline (standard decoding):")
baseline = model.generate(input_ids.to(model.device), max_new_tokens=50, do_sample=False)
print(tokenizer.decode(baseline[0]))

Baseline (standard decoding):
class User:                                                                                                                       
      def __init__(self, name, age):     
          self.name = name                                                                                                                      
          self.age = age                                                                                                                        
                                                                                                                                                
  class Admin:                                              
      def __init__(self, name, age):
          self.name = name                                                                                                                      
          self.age = age
                                                                                              

In [24]:
import time
import torch

# ============================================================
# Test Prompts — varying repetition levels
# ============================================================

PROMPTS = {
    "high_rep_classes": '''class User:
    def __init__(self, name, age):
        self.name = name
        self.age = age

class Admin:
    def __init__(self, name, age):
        self.name = name
        self.age = age

class Guest:
    def __init__(self, name, age):
        self.name = name
        self.age = age

class Member:
    def __init__(self, name, age):''',

    "high_rep_functions": '''def calculate_sum(a, b):
    """Add two numbers."""
    return a + b

def calculate_diff(a, b):
    """Subtract two numbers."""
    return a - b

def calculate_product(a, b):
    """Multiply two numbers."""
    return a * b

def calculate_quotient(a, b):''',

    "high_rep_api": '''@app.route("/users", methods=["GET"])
def get_users():
    users = User.query.all()
    return jsonify([u.to_dict() for u in users]), 200

@app.route("/users/<int:id>", methods=["GET"])
def get_user(id):
    user = User.query.get_or_404(id)
    return jsonify(user.to_dict()), 200

@app.route("/users", methods=["POST"])
def create_user():
    data = request.get_json()
    user = User(**data)
    db.session.add(user)
    db.session.commit()
    return jsonify(user.to_dict()), 201

@app.route("/users/<int:id>", methods=["PUT"])
def update_user(id):''',

    "high_rep_tests": '''import pytest

class TestCalculator:
    def test_add_positive(self):
        assert calculator.add(2, 3) == 5

    def test_add_negative(self):
        assert calculator.add(-2, -3) == -5

    def test_add_zero(self):
        assert calculator.add(0, 0) == 0

    def test_subtract_positive(self):
        assert calculator.subtract(5, 3) == 2

    def test_subtract_negative(self):
        assert calculator.subtract(-2, -3) == 1

    def test_multiply_positive(self):''',

    "medium_rep_validation": '''class UserSerializer:
    def validate_name(self, value):
        if not value:
            raise ValidationError("Name is required")
        if len(value) > 100:
            raise ValidationError("Name too long")
        return value.strip()

    def validate_email(self, value):
        if not value:
            raise ValidationError("Email is required")
        if len(value) > 200:
            raise ValidationError("Email too long")
        return value.strip()

    def validate_phone(self, value):''',

    "low_rep_async": '''import asyncio
import aiohttp

async def fetch_weather(city: str) -> dict:
    async with aiohttp.ClientSession() as session:
        url = f"https://api.weather.com/v1/{city}"
        async with session.get(url) as response:
            return await response.json()

async def main():''',
}


# ============================================================
# Correctness Check
# ============================================================

def verify_correctness(model, tokenizer, prompt, max_new=50):
    """Ensure our decoder produces the same output as greedy baseline."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    # Baseline
    baseline_out = model.generate(input_ids, max_new_tokens=max_new, do_sample=False)
    baseline_text = tokenizer.decode(baseline_out[0], skip_special_tokens=True)

    # Ours
    dec = NGramSpeculativeDecoder(model, tokenizer)
    our_out, _ = dec.generate(input_ids, max_new=max_new)
    our_text = tokenizer.decode(our_out[0], skip_special_tokens=True)

    match = baseline_text == our_text
    if not match:
        # Find first divergence
        for i, (a, b) in enumerate(zip(baseline_out[0].tolist(), our_out[0].tolist())):
            if a != b:
                print(f"  DIVERGE at token {i}: baseline={a} ({tokenizer.decode([a])!r}) "
                      f"vs ours={b} ({tokenizer.decode([b])!r})")
                break
    return match


# ============================================================
# Benchmark
# ============================================================

def benchmark_single(model, tokenizer, prompt, max_new=100, runs=3):
    """Benchmark one prompt across all methods."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    prompt_len = input_ids.shape[1]

    # Warmup
    _ = model.generate(input_ids, max_new_tokens=10, do_sample=False)
    torch.cuda.synchronize()

    results = {}

    # 1. Baseline
    times, gen_tokens = [], 0
    for _ in range(runs):
        torch.cuda.synchronize()
        t = time.perf_counter()
        out = model.generate(input_ids, max_new_tokens=max_new, do_sample=False)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t)
        gen_tokens = out.shape[1] - prompt_len
    results["baseline"] = {"time": sum(times)/runs, "tokens": gen_tokens}

    # 2. Prompt Lookup (HF built-in, flat verification)
    times, gen_tokens = [], 0
    for _ in range(runs):
        torch.cuda.synchronize()
        t = time.perf_counter()
        out = model.generate(input_ids, max_new_tokens=max_new, do_sample=False,
                           prompt_lookup_num_tokens=4)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t)
        gen_tokens = out.shape[1] - prompt_len
    results["prompt_lookup"] = {"time": sum(times)/runs, "tokens": gen_tokens}

    # 3. Ours (n-gram + tree attention)
    times, gen_tokens = [], 0
    last_stats = None
    for _ in range(runs):
        dec = NGramSpeculativeDecoder(model, tokenizer)
        torch.cuda.synchronize()
        t = time.perf_counter()
        out, stats = dec.generate(input_ids, max_new=max_new)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t)
        gen_tokens = out.shape[1] - prompt_len
        last_stats = stats
    results["ngram_tree"] = {
        "time": sum(times)/runs,
        "tokens": gen_tokens,
        "tok_per_step": last_stats.tokens_per_step,
        "spec_rate": last_stats.speculation_rate,
        "accepts": dict(sorted(last_stats.accepted_counts.items())),
    }

    return results, prompt_len


def run_full_benchmark(model, tokenizer, prompts=None, max_new=100, runs=3):
    """Run benchmark across multiple prompts."""
    if prompts is None:
        prompts = PROMPTS

    all_results = {}

    # Step 1: Correctness
    print("=" * 70)
    print("CORRECTNESS CHECK")
    print("=" * 70)
    for name, prompt in prompts.items():
        ok = verify_correctness(model, tokenizer, prompt, max_new=50)
        status = "PASS" if ok else "FAIL"
        print(f"  {name:30s} [{status}]")
    print()

    # Step 2: Benchmarks
    print("=" * 70)
    print("BENCHMARK")
    print("=" * 70)

    for name, prompt in prompts.items():
        print(f"\n--- {name} ---")
        results, prompt_len = benchmark_single(model, tokenizer, prompt, max_new, runs)
        all_results[name] = results

        base_t = results["baseline"]["time"]
        print(f"  Prompt: {prompt_len} tokens | Generate: {max_new} tokens | Runs: {runs}")
        print(f"  {'Method':20s} {'Time':>8s} {'Tok/s':>8s} {'Speedup':>8s} {'Tok/Step':>9s} {'Spec%':>6s}")
        print(f"  {'-'*60}")

        for method, data in results.items():
            t = data["time"]
            tok_s = data["tokens"] / t
            speedup = base_t / t
            tps = f"{data.get('tok_per_step', '-'):>7.2f}" if 'tok_per_step' in data else "       -"
            spec = f"{data.get('spec_rate', 0):>5.0%}" if 'spec_rate' in data else "     -"
            print(f"  {method:20s} {t:>7.3f}s {tok_s:>7.1f} {speedup:>7.2f}x {tps} {spec}")

        if "accepts" in results.get("ngram_tree", {}):
            print(f"  Accept distribution: {results['ngram_tree']['accepts']}")

    # Step 3: Summary
    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"  {'Prompt':30s} {'Baseline':>10s} {'PromptLkp':>10s} {'NgramTree':>10s} {'Tree vs PL':>10s}")
    print(f"  {'-'*70}")

    for name, results in all_results.items():
        base_t = results["baseline"]["time"]
        pl_speedup = base_t / results["prompt_lookup"]["time"]
        nt_speedup = base_t / results["ngram_tree"]["time"]
        tree_vs_pl = results["prompt_lookup"]["time"] / results["ngram_tree"]["time"]
        print(f"  {name:30s} {'1.00x':>10s} {pl_speedup:>9.2f}x {nt_speedup:>9.2f}x {tree_vs_pl:>9.2f}x")

    print("=" * 70)
    return all_results


# ============================================================
# Run it
# ============================================================

all_results = run_full_benchmark(model, tokenizer, max_new=100, runs=3)

CORRECTNESS CHECK
  DIVERGE at token 105: baseline=2 ('#') vs ours=1040 ('class')
  high_rep_classes               [FAIL]
  DIVERGE at token 83: baseline=4828 (' raise') vs ours=470 (' return')
  high_rep_functions             [FAIL]
  DIVERGE at token 171: baseline=2592 ('(data') vs ours=21798 ('(**')
  high_rep_api                   [FAIL]
  high_rep_tests                 [FAIL]
  DIVERGE at token 106: baseline=1372 (' number') vs ours=374 (' is')
  medium_rep_validation          [FAIL]
  low_rep_async                  [PASS]

BENCHMARK

--- high_rep_classes ---
  Prompt: 92 tokens | Generate: 100 tokens | Runs: 3
  Method                   Time    Tok/s  Speedup  Tok/Step  Spec%
  ------------------------------------------------------------
  baseline               3.415s    29.3    1.00x        -      -
  prompt_lookup          2.870s    34.8    1.19x        -      -
  ngram_tree             4.328s    23.6    0.79x    3.09   70%
  Accept distribution: {0: 10, 3: 23}

--- high_rep_f